In [43]:
# !pip install kagglehub

In [44]:
import kagglehub
import pandas as pd
from sklearn.preprocessing import StandardScaler
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader

In [45]:
# Download latest version
path = kagglehub.dataset_download("raminhuseyn/airline-customer-satisfaction")
# print(path)

In [46]:
df = pd.read_csv(path + "/Airline_customer_satisfaction.csv")
df.head()

,satisfaction,Customer Type,Age,Type of Travel,Class,Flight Distance,Seat comfort,Departure/Arrival time convenient,Food and drink,Gate location,...,Online support,Ease of Online booking,On-board service,Leg room service,Baggage handling,Checkin service,Cleanliness,Online boarding,Departure Delay in Minutes,Arrival Delay in Minutes
0,satisfied,Loyal Customer,65,Personal Travel,Eco,265,0,0,0,2,...,2,3,3,0,3,5,3,2,0,0.0
1,satisfied,Loyal Customer,47,Personal Travel,Business,2464,0,0,0,3,...,2,3,4,4,4,2,3,2,310,305.0
2,satisfied,Loyal Customer,15,Personal Travel,Eco,2138,0,0,0,3,...,2,2,3,3,4,4,4,2,0,0.0
3,satisfied,Loyal Customer,60,Personal Travel,Eco,623,0,0,0,3,...,3,1,1,0,1,4,1,3,0,0.0
4,satisfied,Loyal Customer,70,Personal Travel,Eco,354,0,0,0,3,...,4,2,2,0,2,4,2,5,0,0.0


In [47]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [48]:
### DATA PRE-PROCESSING

# 1. See the shape (Rows, Columns)
print(df.shape)

# 2. See which columns have holes (Missing data)
# print(df.isnull().sum())

# 3. Drop null value rows
df = df.dropna()

# Look, if data is biased toward one value? see percentages:
# print(df['satisfaction'].value_counts(normalize=True))

# Convert target to 0 and 1 
df['satisfaction'] = df['satisfaction'].map({'dissatisfied': 0, 'satisfied': 1})
# print(df['satisfaction'].value_counts())

# We have categorical data, for example, business class, eco and eco plus class. The model doesn't understand the difference and even we assign them number like 1,2,3 the model will think 3 is larger than and more important, so we'll split it into different cols. 

# For categorical data with two possible values, we'll convert them to binary and it does not matter, which one will get which one cause nn just need to know the difference. 

categorical_cols = ['Customer Type','Type of Travel','Class']
df = pd.get_dummies(df,columns=categorical_cols,drop_first=True,dtype=int) # drop first will drop the first value alphabatically from each col, for col with three vals, there will be two cols, if both 0, then it means the third class was true. 

# in the dataset we've numerical cols with varying scale, such as between 0-80 while flight distances are large, the model will give more preferences to flight distances, thinking it as more important. 

# we need to standarize them
# Z-score = x - mean / std.dev
scaler = StandardScaler()
num_cols = ['Age','Flight Distance','Departure Delay in Minutes','Arrival Delay in Minutes']
df[num_cols] = scaler.fit_transform(df[num_cols])

df.head()
df.shape

(129880, 22)


(129487, 23)

In [49]:
np.random.seed(100)

In [50]:
# Splitting data
indices = np.arange(len(df))
np.random.shuffle(indices)

train_split = int(0.8 * len(df))
val_split = int(0.9 * len(df))

# Acts as ticket numbers
train_idx = indices[:train_split]
val_idx = indices[train_split:val_split]
test_idx = indices[val_split:]

# 1. Define X (features) and Y (target)
X = df.drop(columns=['satisfaction']).to_numpy()
# print(df.head())
Y = df['satisfaction'].to_numpy()
# print(Y)

# 2. Extract specific sets 
x_train, y_train = X[train_idx], Y[train_idx]
x_val, y_val = X[val_idx], Y[val_idx]
x_test, y_test = X[test_idx], Y[test_idx]

# print(x_train.shape) # (m, num_features)
# we want (num_features, m)
x_train = x_train.T
# print(y_train.shape) # (m,)
# we want (1,m)
y_train = y_train.reshape(1,-1)

x_test = x_test.T
y_test = y_test.reshape(1,-1)

x_val = x_val.T
y_val = y_val.T

x_train.shape
num_features = x_train.shape[0]

In [51]:
class ManualRegression():
    def __init__(self,num_features):
        self.n1 = 16 # neurons in first hidden layer
        self.n2 = 8 #neurons in second hidden layer
        self.n3 = 1 # output layer

        self.w1 = np.random.randn(self.n1, num_features) * 0.01 # 16,22
        self.b1 = np.zeros((self.n1, 1)) # 16,1

        self.w2 = np.random.randn(self.n2, self.n1) * 0.01 # 8,16
        self.b2 = np.zeros((self.n2, 1)) # 8,1

        self.w3 = np.random.randn(self.n3, self.n2) * 0.01 # 1,8
        self.b3 = np.zeros((self.n3, 1)) # 1, 1

    def forward(self,x):
        # Layer 1
        self.z1 = np.dot(self.w1 , x ) + self.b1 # (16,22).(22,m) = 16,m
        self.a1 = np.maximum(0,self.z1) # ReLU

        # Layer 2
        self.z2 = np.dot(self.w2 , self.a1 ) + self.b2 # (8,16).(16,m)= 8,m
        self.a2 = np.maximum(0,self.z2) # ReLU

        # Layer 3 (Sigmoid Output)
        self.z3 = np.dot(self.w3, self.a2) + self.b3 # (1,m).(8,m) = 1,m
        self.a3 = 1 / (1 + np.exp(-self.z3)) # Sigmoid formula
        
        return self.a3
    
    def backward(self,x,y):
        # we don't loss here, cause dz3 math simplified to the follwing equation
        self.m = x.shape[1]

        # layer 3 
        self.dz3 = self.a3 - y # a3 contains model predictions between 0 and 1, (1,m)
        self.dw3 = (1/self.m) * np.dot(self.dz3,self.a2.T) # (1,m).(m,8) = 1,8
        self.db3 = (1/self.m) * np.sum(self.dz3,axis=1,keepdims=True) # 1,1

        # layer 2 
        self.dz2 = np.dot(self.w3.T,self.dz3) *  np.where(self.z2 > 0, 1,0) # (8,1).(1,m) = 8,m
        #Derivative of ReLU, also element-wise multiplication to get the dimensions right. we do same for output layer sigmoid but it wasn't needed due to math simplification 
        self.dw2 = (1/self.m) * np.dot(self.dz2,self.a1.T) # (8,m).(m,16) = 8,16
        self.db2 = (1/self.m) * np.sum(self.dz2,axis=1,keepdims=True) # 8,1

        # layer 1
        self.dz1 = np.dot(self.w2.T,self.dz2) *  np.where(self.z1 > 0, 1,0) # (16,8).(8,m) = 16,m
        self.dw1 = (1/self.m) * np.dot(self.dz1,x.T) # (16,m).(m,22) = 16,22
        self.db1 = (1/self.m) * np.sum(self.dz1,axis=1,keepdims=True) # 16,1

    def update_parameters(self, lr):
        # Layer 1 updates
        self.w1 = self.w1 - lr * self.dw1
        self.b1 = self.b1 - lr * self.db1
        
        # Layer 2 updates
        self.w2 = self.w2 - lr * self.dw2
        self.b2 = self.b2 - lr * self.db2
        
        # Layer 3 updates
        self.w3 = self.w3 - lr * self.dw3
        self.b3 = self.b3 - lr * self.db3

In [58]:
train_losses = []
val_losses = []

# Initialize your manual network
model = ManualRegression(num_features=num_features)
lr = 0.5

for epoch in range(1000):
    # --- TRAINING PHASE ---
    # 1. Forward pass on training data
    yHat_train = model.forward(x_train)
    
    # 2. Vectorized Log Loss (safeguarded against log(0) with np.clip)
    yHat_train = np.clip(yHat_train, 1e-15, 1 - 1e-15)
    train_loss = -np.mean(y_train * np.log(yHat_train) + (1 - y_train) * np.log(1 - yHat_train))
    train_losses.append(train_loss)
    
    # 3. Backward pass & Parameter Update
    model.backward(x_train, y_train)
    model.update_parameters(lr=lr)
    
    # --- VALIDATION PHASE ---
    # 4. Forward pass on validation data (No backward or update here!)
    yHat_val = model.forward(x_val)
    yHat_val = np.clip(yHat_val, 1e-15, 1 - 1e-15)
    val_loss = -np.mean(y_val * np.log(yHat_val) + (1 - y_val) * np.log(1 - yHat_val))
    val_losses.append(val_loss)
    
    # Print progress every 10 epochs
    if epoch % 10 == 0 or epoch == 99:
        print(f"Epoch {epoch} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

Epoch 0 | Train Loss: 0.6931 | Val Loss: 0.6920
Epoch 10 | Train Loss: 0.6889 | Val Loss: 0.6884
Epoch 20 | Train Loss: 0.6886 | Val Loss: 0.6881
Epoch 30 | Train Loss: 0.6886 | Val Loss: 0.6881
Epoch 40 | Train Loss: 0.6886 | Val Loss: 0.6881
Epoch 50 | Train Loss: 0.6886 | Val Loss: 0.6881
Epoch 60 | Train Loss: 0.6886 | Val Loss: 0.6881
Epoch 70 | Train Loss: 0.6886 | Val Loss: 0.6881
Epoch 80 | Train Loss: 0.6886 | Val Loss: 0.6881
Epoch 90 | Train Loss: 0.6886 | Val Loss: 0.6881
Epoch 99 | Train Loss: 0.6886 | Val Loss: 0.6881
Epoch 100 | Train Loss: 0.6886 | Val Loss: 0.6881
Epoch 110 | Train Loss: 0.6886 | Val Loss: 0.6881
Epoch 120 | Train Loss: 0.6886 | Val Loss: 0.6881
Epoch 130 | Train Loss: 0.6886 | Val Loss: 0.6881
Epoch 140 | Train Loss: 0.6886 | Val Loss: 0.6881
Epoch 150 | Train Loss: 0.6886 | Val Loss: 0.6881
Epoch 160 | Train Loss: 0.6886 | Val Loss: 0.6881
Epoch 170 | Train Loss: 0.6886 | Val Loss: 0.6881
Epoch 180 | Train Loss: 0.6886 | Val Loss: 0.6881
Epoch 190 | 

In [53]:
# # Shuffle and split data
# np.random.seed(100)
# indices = np.arange(len(df))
# np.random.shuffle(indices)

# train_split = int(0.8 * len(df))
# val_split = int(0.9 * len(df))

# # Acts as ticket numbers
# train_idx = indices[:train_split]
# val_idx = indices[train_split:val_split]
# test_idx = indices[val_split:]

# # 1. Define X (features) and Y (target)
# X = df.drop(columns=['satisfaction']).to_numpy()
# # print(df.head())
# Y = df['satisfaction'].to_numpy()
# # print(Y)

# # 2. Extract specific sets 
# x_train, y_train = X[train_idx], Y[train_idx]
# x_val, y_val = X[val_idx], Y[val_idx]
# x_test, y_test = X[test_idx], Y[test_idx]

# # print(X.shape)

# num_features = x_train.shape[1]

# x_train_tensor = torch.as_tensor(x_train,dtype=torch.float).to(device).reshape(-1,num_features)
# y_train_tensor = torch.as_tensor(y_train,dtype=torch.float).to(device).reshape(-1,1)
# x_val_tensor = torch.as_tensor(x_val,dtype=torch.float).to(device).reshape(-1,num_features)
# y_val_tensor = torch.as_tensor(y_val,dtype=torch.float).to(device).reshape(-1,1)  # pytorch linear layer expects (batch_size, features)

# # Create the training and validation datasets
# train_ds = TensorDataset(x_train_tensor, y_train_tensor) # Pairs up input and output
# val_ds = TensorDataset(x_val_tensor, y_val_tensor)


# # DataLoader act as conveyer belt, instead of feeding all rows at once, it sends data in batches, which helps model help update weights multiple times (SGD), also this speeds up the math/ease up the math/calculations
# train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
# val_loader = DataLoader(val_ds, batch_size=64, shuffle=False) # Don't need to shuffle for validation, shuffling needed only for training. 

# print("X shape", X.shape)
# print("x-train shape", x_train.shape)
# print("x_train_tensor shape", x_train_tensor.shape)


In [54]:
# class AirlineModel(nn.Module):
#     # Define the model architecture
#     def __init__(self, input_size):
#         super().__init__()
#         self.hidden1 = nn.Linear(input_size,16) # input to 16 neurons in first layer
#         self.hidden2 = nn.Linear(16,8) # 16 inputs from first layer and 8 neurons in 2nd layer
#         self.output = nn.Linear(8,1) # 8 inputs from layer 2 and 1 output. 

#         # Total parameters
#         # w = input_size * out_features/neurons, e.g first layer if input is 16, weights for layer 1 are 16 x 16
#         # b = one bias for each neuron, e.g for first layer 16 biases.

#         # Activations
#         self.ReLU =  nn.ReLU()
#         self.tanH = nn.Tanh()
#         self.sigmoid = nn.Sigmoid()

#     # math flow in forward
#     def forward(self, x):
#         x = self.hidden1(x)
#         x = self.ReLU(x)

#         x = self.hidden2(x)
#         x = self.tanH(x)

#         x = self.output(x)
#         x = self.sigmoid(x)

#         return x
# #

In [55]:
# # initialize model 
# model = AirlineModel(num_features).to(device)            